# 01 — Data & Preprocessing

Loading data, splitting, scaling, encoding, and imputing — the groundwork before any model touches the data.

## Loading Data

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris, fetch_california_housing, make_classification

# built-in toy/real datasets
iris = load_iris(as_frame=True)
X, y = iris.data, iris.target

housing = fetch_california_housing(as_frame=True)   # downloads on first use

# synthetic data for quick experiments
X_synth, y_synth = make_classification(n_samples=1000, n_features=20, n_informative=10, random_state=42)

df = pd.read_csv("data.csv")   # typical real-world entry point


## Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y   # stratify preserves class proportions
)


## Scaling (numerical features)

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, Normalizer

scaler = StandardScaler()          # zero mean, unit variance — most common default
X_train_scaled = scaler.fit_transform(X_train)   # fit only on train
X_test_scaled = scaler.transform(X_test)          # transform test with train's stats

MinMaxScaler(feature_range=(0, 1))   # scales to a fixed range
RobustScaler()                        # uses median/IQR, robust to outliers
Normalizer()                          # scales each SAMPLE (row) to unit norm, not each feature


## Encoding Categorical Features

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder

cat_df = pd.DataFrame({"color": ["red", "blue", "green", "blue"]})

ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
ohe.fit_transform(cat_df)          # one column per category

oe = OrdinalEncoder()               # single integer column, implies an order — use only if categories ARE ordinal
oe.fit_transform(cat_df)

le = LabelEncoder()                 # for encoding the TARGET y, not features
y_encoded = le.fit_transform(["cat", "dog", "cat"])


## Imputing Missing Values

In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer

num_imputer = SimpleImputer(strategy="mean")       # or "median", "most_frequent", "constant"
X_train_imputed = num_imputer.fit_transform(X_train)

cat_imputer = SimpleImputer(strategy="most_frequent")

knn_imputer = KNNImputer(n_neighbors=5)   # imputes using similar rows, more accurate but slower


## Polynomial & Interaction Features

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_train[:, :3])   # generates x1^2, x1*x2, etc.


## Discretization / Binning

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

kbd = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile")
kbd.fit_transform(X_train[:, [0]])


## Handling Mixed Numerical + Categorical Data (preview — full detail in notebook 02)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_features = ["age", "income"]
categorical_features = ["city"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])


## Common Gotchas
- Always `fit` scalers/encoders on the TRAINING set only, then `transform` (never `fit`) the test set — fitting on test data leaks information.
- `OneHotEncoder(handle_unknown="ignore")` prevents errors when test data has unseen categories.
- `LabelEncoder` is for target labels; use `OneHotEncoder`/`OrdinalEncoder` for feature columns.
- Imputing before splitting also leaks information — impute after the split, fit on train only.